In [ ]:
#| default_exp search

# Search

> MCTS API. Monte Carlo Tree Search is a random sampling algorithm. AlphaZero modifies MCTS to employ NNATS/DLATS: deeplearning augmented tree search. 

In [ ]:
#|hide
from nbdev.showdoc import *

In [ ]:
#| export
import numpy as np
import alphazero.go as go

In [ ]:
#| export
class MCTSNode():
    def __init__(self, state, action_probs, valid_actions, branches, value, is_endstate, parent_index, index):
        self.action_probs = action_probs
        self.valid_actions = valid_actions
        self.branches = branches
        self.visits = 0
        self.value = value
        self.path_value = value
        self.is_endstate = is_endstate
        self.parent_index = parent_index
        self.index = index
        self.state = state

In [ ]:
engine = go.Position()


In [ ]:
valid_actions = engine.all_legal_moves()

In [ ]:
len(valid_actions)

362

In [ ]:
rng = np.random.default_rng()

state_size=10
n_actions = 4
n_sims = 100
root_idx = 0
tree = {'counter':root_idx+1}

# initialize the root node of the search tree
tree[root_idx] = MCTSNode(state=rng.integers(0,n_actions, size=(state_size,state_size)),
                         action_probs=rng.random(size=n_actions),
                         valid_actions=rng.integers(0, 2, size=n_actions),
                         branches=[tree['counter']+i for i in range(n_actions)],
                         value=rng.random(),
                         is_endstate=False,
                         parent_index=None,
                         index=0)
tree['counter'] += n_actions


In [ ]:
# sample simulation
for i in range(10):
    # reset to root node on sim start
    select_idx = parent_idx = root_idx
    value = 0.0
    
    # loop until find end or new state
    while tree.get(select_idx, None) is not None and not tree[select_idx].is_endstate:
        # select branch
        # parent idx keeps track of current node iot to specify a selected nodes parent on creation
        parent_idx = select_idx
        select_idx = rng.choice(tree[select_idx].branches)

    # if new state: expand tree
    if not tree.get(select_idx, None):
        tree[select_idx] = MCTSNode(state=rng.integers(0,n_actions, size=(state_size,state_size)),
                                    action_probs=rng.random(size=n_actions),
                                    valid_actions=rng.integers(0, 2, size=n_actions),
                                    branches=[tree['counter']+i for i in range(n_actions)],
                                    value=rng.random(),
                                    is_endstate=False,
                                    parent_index=parent_idx,
                                    index=select_idx)
        tree['counter'] += n_actions

    # in all cases: propagate value back down path to root
    value = tree[select_idx].value
    while parent_idx is not None:
        tree[parent_idx].path_value += value
        select_idx = parent_idx
        parent_idx = tree[select_idx].parent_index
